# LisPy en Jupyter Notebook

**LisPy** es un intérprete de **ANSI Common Lisp** con un puente **FFI a CPython** embebido.

Usa el prefijo `py:` para llamar a Python desde Lisp. El intérprete mantiene **todo el estado** entre celdas: variables Lisp, funciones, macros, módulos Python importados y el namespace `__main__` de Python.

En este notebook se cargan automaticamente las librerias: `ffi`, `continuations`, `defpyfun`, `defpyclass`, `python-template` y `persist`. Las librerias pesadas (`numpy-fast`, `sympy-bridge`, `defmath`) se cargan bajo demanda en sus secciones.

## 1. Aritmetica y Funciones

In [ ]:
(+ 1 2 3 4 5)

In [ ]:
(sqrt (+ (expt 3 2) (expt 4 2)))

In [ ]:
(defun factorial (n)
  (if (<= n 1) 1 (* n (factorial (- n 1)))))
(factorial 10)

In [ ]:
(defun fibonacci (n)
  (if (<= n 1) n (+ (fibonacci (- n 1)) (fibonacci (- n 2)))))
(fibonacci 20)

## 2. Estado persistente entre celdas

Las funciones definidas en una celda estan disponibles en las siguientes.

In [ ]:
(factorial 5)

In [ ]:
(mapcar #'factorial '(1 2 3 4 5 6 7))

In [ ]:
(defparameter *contador* 0)

(defun incrementar ()
  (incf *contador*)
  (format nil "Contador: ~D" *contador*))

In [ ]:
(incrementar)

In [ ]:
(incrementar)

## 3. Macros y Programacion Funcional

In [ ]:
(defmacro when-positive (expr &body body)
  `(let ((val ,expr))
     (when (> val 0)
       ,@body)))
(when-positive (- 10 3)
  (print "Es positivo")
  (* 2 2))

In [ ]:
(mapcar (lambda (x) (* x x)) '(1 2 3 4 5))

In [ ]:
(reduce #'+ '(1 2 3 4 5 6 7 8 9 10))

## 4. Python FFI — Conceptos basicos

Los simbolos en el paquete `py:` se auto-resuelven contra modulos y builtins de Python.
El cache de modulos es **persistente** entre celdas.

In [ ]:
(py:abs -42)

In [ ]:
(py:sum (list 1 2 3 4 5 6 7 8 9 10))

In [ ]:
(py:max 3.14 2.71 1.61 42)

### 4.1 Importando modulos (persistente entre celdas)

`py:import` guarda el modulo en el cache. En celdas posteriores, el simbolo se resuelve automaticamente sin re-importar.

In [ ]:
(py:import "math")

In [ ]:
(py:attr math "pi")

In [ ]:
(py:call (py:attr math "sqrt") 16)

In [ ]:
(py:call (py:attr math "cos") 0)

### 4.2 py:run — Ejecutar codigo Python arbitrario

El codigo ejecutado con `py:run` corre en `__main__` de Python. Variables, funciones e imports definidos aqui persisten para celdas posteriores.

In [ ]:
(py:run "
import datetime
ahora = datetime.datetime.now()
print(f'Hoy: {ahora:%Y-%m-%d %H:%M:%S}')
")

In [ ]:
(py:run "
print(f'La variable \"ahora\" sigue viva: {ahora}')
print(f'Tipo: {type(ahora).__name__}')
")

In [ ]:
(py:run "
import json
datos = {'nombre': 'LisPy', 'version': 1.0, 'lenguaje': 'Common Lisp + Python'}
print(json.dumps(datos, indent=2))
")

In [ ]:
(py:run "
# datos y json siguen en __main__ desde la celda anterior
print(f'Claves: {list(datos.keys())}')
print(f'JSON disponible: {json is not None}')
")

### 4.3 NumPy — Computacion numerica con estado persistente

In [ ]:
(py:import "numpy")

In [ ]:
(let ((a (py:call (py:attr numpy "array") (list 1 2 3 4 5))))
  (list a
        (py:call (py:attr numpy "mean") a)
        (py:call (py:attr numpy "std") a)))

In [ ]:
(py:call (py:attr numpy "linspace") 0 pi 5)

### 4.4 Auto-resolucion de simbolos Python

Al escribir `math`, `numpy` o cualquier nombre de modulo/builtin directamente, LisPy busca en:
1. Cache de modulos Lisp
2. `sys.modules` de Python
3. `builtins` de Python
4. `__main__` de Python (variables definidas con `py:run`)

In [ ]:
math

In [ ]:
(py:call (py:attr math "factorial") 10)

In [ ]:
numpy

## 5. CLOS — Common Lisp Object System

In [ ]:
(defclass point ()
  ((x :initarg :x :accessor point-x)
   (y :initarg :y :accessor point-y)))

(defmethod distance-from-origin ((p point))
  (sqrt (+ (expt (point-x p) 2) (expt (point-y p) 2))))

(let ((p (make-instance 'point :x 3 :y 4)))
  (distance-from-origin p))

In [ ]:
(let ((p (make-instance 'point :x 6 :y 8)))
  (list :coordenadas (list (point-x p) (point-y p))
        :distancia (distance-from-origin p)))

## 6. Macro de Lectura `#_` — Acceso compacto a Python

El macro de lectura `#_` convierte `#_modulo:atributo` en `(py:%chain MODULO ATRIBUTO)`.
Permite acceder a modulos, atributos y metodos de Python con una sintaxis minimalista.

Tambien se puede invocar directamente con parentesis: `#_math:sqrt(16)` → `(py:call (py:%chain MATH SQRT) 16)`.

El punto `.` funciona como separador alternativo a `:`: `@numpy.linspace`.

In [ ]:
;; Acceso compacto a atributos
#_math:pi
#_math:sqrt(16)
#_builtins:sum(py:->python '(1 2 3 4 5))

In [ ]:
;; Encadenamiento con punto
#_numpy.linspace(0 pi 10)

In [ ]:
;; Comparacion con la forma clasica
;; py:call (py:attr math "sqrt") 16   ; clasica
;; #_math:sqrt(16)                     ; compacta

## 7. Conversion Explicita y Argumentos Keyword

### 7.1 `py:->lisp` y `py:->python`

La conversion automatica cubre tipos basicos. Para control manual:
- `py:->python` convierte un valor Lisp a `VALUE_PYTHON`
- `py:->lisp` convierte `VALUE_PYTHON` a tipo Lisp nativo

### 7.2 `py:call-kw` — Argumentos keyword

Permite pasar argumentos keyword (kwargs) a funciones Python usando una hash-table Lisp.

In [ ]:
;; py:->lisp: convertir objeto Python a Lisp nativo
(let ((py-dict (py:run "
__lispy_tmp = {'a': 1, 'b': 2, 'c': 3}
")))
  (declare (ignore py-dict))
  (py:->lisp (py:attr (py:import "__main__") "__lispy_tmp")))

In [ ]:
;; py:->python: convertir Lisp a objeto Python
(py:->python (list 1 2 3 4 5))
(py:type (py:->python (list 1 2 3 4 5)))

In [ ]:
;; py:call-kw con hash-table de keywords
(let ((kwargs (make-hash-table)))
  (setf (gethash "sep" kwargs) " | "
        (gethash "end" kwargs) " !!\n")
  (py:call-kw (py:attr (py:import "builtins") "print")
              (list "a" "b" "c")
              kwargs))

In [ ]:
;; py:getitem — indexar objetos Python
(let ((d (py:run "
__lispy_d = {'nombre': 'LisPy', 'version': 1.0}
")))
  (declare (ignore d))
  (let ((d (py:attr (py:import "__main__") "__lispy_d")))
    (list (py:getitem d "nombre")
          (py:getitem d "version"))))

## 8. Depuracion Python desde Lisp

LisPy expone herramientas de introspeccion y depuracion sobre el runtime Python:

| Funcion | Descripcion |
|---------|-------------|
| `py:last-traceback` | Lista de frames del ultimo error Python |
| `py:frame-locals` | Hash-table con variables locales de un frame |
| `py:frame-globals` | Hash-table con variables globales de un frame |
| `py:frame-eval` | Evaluar expresion en contexto de un frame |
| `py:help` | `help(obj)` de Python |
| `py:dir` | `dir(obj)` de Python |
| `py:type` | `type(obj)` de Python |
| `py:repr` | `repr(obj)` de Python |

In [ ]:
;; py:help — documentacion Python desde Lisp
(py:help (py:import "math"))

In [ ]:
;; py:dir — listar atributos (primeros 10 de math)
(let ((attrs (py:dir (py:import "math"))))
  (subseq attrs 0 (min 10 (length attrs))))

In [ ]:
;; py:type y py:repr
(let ((obj (py:import "math")))
  (list :type (py:type obj)
        :repr (py:repr obj)))

In [ ]:
;; py:last-traceback — inspeccionar el ultimo error Python
(ignore-errors
  (py:run "1/0"))
(let* ((frames (py:last-traceback))
       (last-frame (first (last frames))))
  (when last-frame
    (list :archivo (gethash "filename" last-frame)
          :linea (gethash "lineno" last-frame)
          :funcion (gethash "name" last-frame))))

## 9. Callbacks Lisp→Python

`py:register-callback` expone una funcion Lisp como callback Python.
La funcion queda registrada en `builtins` de Python con el nombre `lispy_callback_NOMBRE`.

Esto permite que codigo Python ejecutado con `py:run` llame de vuelta a funciones Lisp.

In [ ]:
;; Definir una funcion Lisp y registrarla como callback
(defun mi-suma (a b)
  (+ a b))

(py:register-callback 'mi-suma)

In [ ]:
;; Usar el callback desde Python
(py:run "
resultado = lispy_callback_mi_suma(10, 20)
print(f'Lisp dice: 10 + 20 = {resultado}')
")

## 10. Serializacion con pickle

`py:pickle-dumps` serializa un objeto Python a string hexadecimal.
`py:pickle-loads` reconstruye el objeto desde el string.

In [ ]:
;; Serializar un objeto Python
(let* ((obj (py:->python (list 1 2 3 (list 4 5 6))))
       (hex (py:pickle-dumps obj)))
  (format nil "Serializado (~D bytes): ~A..." (length hex) (subseq hex 0 40)))

In [ ]:
;; Deserializar de vuelta a Lisp
(let* ((hex (py:pickle-dumps (py:->python (list :a 1 :b 2 :c 3))))
       (obj (py:pickle-loads hex)))
  (py:->lisp obj))

## 11. FFI Nativo a C (`lib/ffi.lp`)

La libreria `ffi.lp` permite llamar funciones C desde DLLs/.so/.dylib sin recompilar LisPy,
usando `ctypes` de Python como puente. Se carga automaticamente al iniciar el kernel.

**Tipos soportados:** `:void`, `:bool`, `:char`, `:byte`, `:short`, `:ushort`, `:int`, `:uint`,
`:long`, `:ulong`, `:longlong`, `:ulonglong`, `:float`, `:double`, `:pointer`, `:string`, `:wstring`.

In [ ]:
;; Cargar msvcrt.dll (Windows) y definir wrappers para funciones C
(ffi:with-ffi "msvcrt.dll"
  (c-sqrt :double (:double) "sqrt")
  (c-sin  :double (:double) "sin")
  (c-cos  :double (:double) "cos")
  (c-pow  :double (:double :double) "pow")
  (c-abs  :int (:int) "abs"))

In [ ]:
;; Usar las funciones C como si fueran Lisp
(list :sqrt-64 (c-sqrt 64.0)
      :pow-2-10 (c-pow 2.0 10.0)
      :sin-0 (c-sin 0.0)
      :cos-pi (c-cos pi)
      :abs-minus-42 (c-abs -42))

In [ ]:
;; Llamada directa sin definir wrapper (ffi:call)
(ffi:call "msvcrt.dll" "sqrt" :double '(:double) 144.0)

**Nota para Linux/macOS:** Cambia `"msvcrt.dll"` por `"libm.so.6"` (Linux) o `"libSystem.B.dylib"` (macOS).

```lisp
;; Linux
(ffi:with-ffi "libm.so.6"
  (c-sqrt :double (:double) "sqrt"))
```

## 12. NumPy Acelerado (`lib/numpy-fast.lp`)

La libreria `numpy-fast.lp` ofrece operaciones vectoriales de NumPy con sintaxis Lisp.
Incluye el macro `defnp` para definir funciones vectorizadas.

Carganos la libreria explicitamente (es pesada, importa NumPy).

In [ ]:
(load "lib/numpy-fast.lp")

In [ ]:
;; Crear arrays y operar
(let* ((a (numpy-arange 0 10 1))
       (b (numpy-sin a)))
  (list :a a :sin-a b))

In [ ]:
;; Funciones estadisticas
(let ((datos (numpy-array (list 10 20 30 40 50 60 70 80 90 100))))
  (list :media (numpy-mean datos)
        :suma (numpy-sum datos)))

In [ ]:
;; defnp: definir funcion vectorizada automaticamente
(defnp raiz-cuadrada (x)
  (sqrt x))

(raiz-cuadrada (numpy-arange 1 6 1))

## 13. SymPy — Matematica Simbolica (`lib/sympy-bridge.lp`)

La libreria `sympy-bridge.lp` conecta LisPy con SymPy para calculo simbolico:
derivadas, integrales, simplificacion algebraica y exportacion LaTeX.

In [ ]:
(load "lib/sympy-bridge.lp")

In [ ]:
;; Crear simbolos y expresiones
(let ((x (sympy-symbol "x"))
      (y (sympy-symbol "y")))
  (declare (ignore x y))
  (list :expr (sympy-expr (+ (expt x 2) (* 3 x) 2))))

In [ ]:
;; Derivada e integral simbolicas
(let ((x (sympy-symbol "x")))
  (let ((expr (sympy-expr (+ (expt x 3) (* 2 (expt x 2)) x 5))))
    (list :derivada (sympy-derive expr x)
          :integral (sympy-integrate expr x))))

In [ ]:
;; Simplificacion y evaluacion numerica
(let ((x (sympy-symbol "x")))
  (let ((expr (sympy-simplify (sympy-expr (+ (* 2 x) (* 3 x))))))
    (list :simplificado expr
          :evalf-x=3 (sympy-evalf expr x 3))))

In [ ]:
;; Exportar a LaTeX
(let ((x (sympy-symbol "x")))
  (sympy-latex (sympy-expr (/ (+ (expt x 2) 1) (- x 1)))))

## 14. Matematicas Hibridas (`lib/defmath.lp`)

La libreria `defmath.lp` define el macro `defmath` que genera automaticamente
dos versiones de una funcion matematica:
- `nombre-sym`: version simbolica (SymPy)
- `nombre-num`: version numerica vectorizada (NumPy)

In [ ]:
(load "lib/defmath.lp")

In [ ]:
;; Definir una funcion hibrida
(defmath cuadratica (x)
  (+ (* 2 (expt x 2)) (* 3 x) 1))

In [ ]:
;; Version numerica (NumPy, vectorizada)
(let ((xs (numpy-linspace 0 5 6)))
  (cuadratica-num xs))

In [ ]:
;; Version simbolica (SymPy)
(let ((x (sympy-symbol "x")))
  (list :expresion (cuadratica-sym x)
        :derivada (sympy-derive (cuadratica-sym x) x)))

## 15. Persistencia de Sesion (`lib/persist.lp`)

La libreria `persist.lp` permite guardar y restaurar sesiones completas de Python (`__main__`).
Todas las variables, modulos importados y objetos definidos con `py:run` se serializan via pickle.

Se carga automaticamente al iniciar el kernel.

In [ ]:
;; Crear datos valiosos en __main__
(py:run "
datos_importantes = {
    'resultados': [42, 73, 99],
    'config': {'debug': True, 'version': '2.0'},
    'modelo_entrenado': True
}
")

;; Guardar la sesion
(py:save-session "mi-sesion.txt")

In [ ]:
;; Verificar que el archivo existe
(let ((path (py:attr (py:import "os") "path")))
  (list :existe (py:call (py:attr path "exists") "mi-sesion.txt")
        :tamano (py:call (py:attr path "getsize") "mi-sesion.txt")))

In [ ]:
;; Restaurar la sesion (simulando un nuevo kernel/notebook)
;; py:load-session "mi-sesion.txt"

;; Verificar los datos restaurados
(py:run "
if 'datos_importantes' in dir():
    print('Sesion restaurada correctamente:')
    print(f'  Resultados: {datos_importantes["resultados"]}')
    print(f'  Config: {datos_importantes["config"]}')
else:
    print('Sesion no restaurada (esperado si no se ejecuto py:load-session)')
")

## 16. Sintaxis Hibrida Lisp/Python (`py:load-hybrid`)

LisPy permite escribir archivos `.lp` que mezclan Lisp y Python en un mismo archivo,
sin necesidad de `py:run`. El preprocesador hibrido separa automaticamente los bloques
Lisp (lineas que empiezan con `(`) de los bloques Python.

Ejemplo de archivo `ejemplo.lp`:
```lisp
(defun saludar (nombre)
  (format nil "Hola, ~A!" nombre))

x = 42
print(f"Python dice: x = {x}")
print(f"Lisp saluda: {saludar('Mundo')}")
```

Para cargarlo: `(py:load-hybrid "ejemplo.lp")` o simplemente `(load "ejemplo.lp")` ya que LisPy detecta archivos hibridos automaticamente.

In [ ]:
;; Crear un archivo hibrido desde el notebook
(py:run "
with open('demo-hibrido.lp', 'w') as f:
    f.write('''(defun doble (n) (* 2 n))

x = 21
print(f"Python: el doble de {x} es {doble(x)}")
''')
print('Archivo demo-hibrido.lp creado')
")

In [ ]:
;; Cargar el archivo hibrido (load ya lo detecta)
(load "demo-hibrido.lp")

In [ ]:
;; Verificar que la funcion Lisp definida en el hibrido funciona
(doble 10)
(doble 3.5)

---
**Recursos:**
- `docs/LisPy-Reference.md` — Documentacion completa del FFI Python (25 secciones)
- `lib/` — Librerias: `ffi`, `numpy-fast`, `sympy-bridge`, `defmath`, `continuations`, `persist`, `python-template`, `defpyfun`, `defpyclass`, `import-hooks`
- `docs/LisPy-Handbook/` — Manual completo en PDF (81 secciones, 390+ ejemplos, 120 problemas)
- `examples/` — Scripts de ejemplo con sintaxis hibrida (101.lp – 110.lp)